# Análise Financeira — Banco Master S/A (2020–2024)

**Tema:** Crescimento acelerado e riscos de solvência  
**Fonte:** BCB/IF.data + Demonstrações Financeiras publicadas  
**Período:** 2020-S1 a 2024-S2


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

df = pd.read_csv('../data/banco_master_processado.csv')
df['data'] = pd.to_datetime(
    df['periodo'].str.replace('S1','06').str.replace('S2','12'), format='%Y-%m', errors='coerce')
print(f"Dataset: {len(df)} períodos — {df['periodo'].min()} a {df['periodo'].max()}")
df[['periodo','ativo_total_bi','carteira_credito_bi','lucro_liquido_mi','alavancagem','indice_basileia_pct']]


## 1. Crescimento do Balanço

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.plot(df['periodo'], df['ativo_total_bi'], marker='o', color='#0066cc', lw=2.5, label='Ativo Total')
ax.plot(df['periodo'], df['carteira_credito_bi'], marker='s', color='#e67e22', lw=2, label='Crédito')
ax.plot(df['periodo'], df['depositos_totais_bi'], marker='^', color='#27ae60', lw=2, label='Depósitos')
ax.set_title('Evolução do Balanço (R$ bilhões)')
ax.tick_params(axis='x', rotation=45)
ax.legend(); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
base = df.iloc[0]
ax2.plot(df['periodo'], df['ativo_total_bi'] / base.ativo_total_bi * 100, marker='o', color='#0066cc', lw=2.5, label='Ativo')
ax2.plot(df['periodo'], df['carteira_credito_bi'] / base.carteira_credito_bi * 100, marker='s', color='#e67e22', lw=2, label='Crédito')
ax2.plot(df['periodo'], df['patrimonio_liquido_bi'] / base.patrimonio_liquido_bi * 100, marker='^', color='#8e44ad', lw=2, label='PL')
ax2.axhline(100, color='gray', lw=1, ls='--', alpha=0.5)
ax2.set_title('Crescimento Acumulado (base 2020-S1 = 100)')
ax2.tick_params(axis='x', rotation=45)
ax2.legend(); ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
crescimento = (df.iloc[-1].ativo_total_bi / df.iloc[0].ativo_total_bi - 1) * 100
print(f"Crescimento total do ativo: {crescimento:.0f}%")


## 2. Indicadores de Risco

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(df['periodo'], df['indice_basileia_pct'], marker='o', color='#c0392b', lw=2.5)
ax.axhline(11, color='red', ls='--', lw=1.5, label='Mínimo (11%)')
ax.set_title('Índice de Basileia (%)'); ax.set_ylim(9, 17)
ax.tick_params(axis='x', rotation=45); ax.legend(); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
cores = ['#e74c3c' if v > 14 else '#3498db' for v in df['alavancagem']]
ax2.bar(df['periodo'], df['alavancagem'].round(1), color=cores, alpha=0.85)
ax2.set_title('Alavancagem (Ativo / PL)')
ax2.tick_params(axis='x', rotation=45); ax2.grid(axis='y', alpha=0.3)

ax3 = axes[2]
ax3.fill_between(range(len(df)), df['razao_credito_deposito'], alpha=0.3, color='#e67e22')
ax3.plot(df['periodo'], df['razao_credito_deposito'], marker='o', color='#e67e22', lw=2)
ax3.set_xticks(range(len(df))); ax3.set_xticklabels(df['periodo'], rotation=45)
ax3.set_title('Razão Crédito / Depósitos'); ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()


## 3. Lucratividade

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bars = ax.bar(df['periodo'], df['lucro_liquido_mi'], color='#27ae60', alpha=0.85)
for bar, val in zip(bars, df['lucro_liquido_mi']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{val:.0f}', ha='center', va='bottom', fontsize=8.5)
ax.set_title('Lucro Líquido (R$ mi)'); ax.tick_params(axis='x', rotation=45); ax.grid(axis='y', alpha=0.3)

ax2 = axes[1]
ax2.plot(df['periodo'], df['roe_pct'], marker='o', color='#8e44ad', lw=2.5, label='ROE (%)')
ax2b = ax2.twinx()
ax2b.plot(df['periodo'], df['margem_lucro_pct'], marker='s', color='#e74c3c', lw=2, ls='--', label='Margem (%)')
ax2.set_title('ROE e Margem Líquida (%)')
ax2.tick_params(axis='x', rotation=45); ax2.grid(axis='y', alpha=0.3)
h1, l1 = ax2.get_legend_handles_labels(); h2, l2 = ax2b.get_legend_handles_labels()
ax2.legend(h1+h2, l1+l2, fontsize=9)

plt.tight_layout(); plt.show()


## 4. Predição — Regressão Linear

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

X = np.arange(len(df)).reshape(-1, 1)
y = df['ativo_total_bi'].values
model = LinearRegression().fit(X, y)

X_fut = np.array([[10], [11]])
y_fut = model.predict(X_fut)
std_res = np.std(y - model.predict(X))
ic_sup = y_fut + 1.96 * std_res
ic_inf = y_fut - 1.96 * std_res

print(f"R² = {r2_score(y, model.predict(X)):.4f}")
print(f"MAE = R$ {mean_absolute_error(y, model.predict(X)):.2f} bi")
print(f"Coeficiente = {model.coef_[0]:.2f} R$ bi/semestre")
print()
for p, proj, inf, sup in zip(['2025-S1','2025-S2'], y_fut, ic_inf, ic_sup):
    print(f"  {p}: R$ {proj:.1f} bi  [IC95%: {inf:.1f} – {sup:.1f}]")

periodos_all = list(df['periodo']) + ['2025-S1', '2025-S2']
y_all = model.predict(np.arange(12).reshape(-1,1))

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df['periodo'], y, 'o-', color='#0066cc', lw=2.5, label='Histórico real')
ax.plot(periodos_all, y_all, '--', color='#e74c3c', lw=2, label='Regressão')
ax.fill_between(['2025-S1','2025-S2'], ic_inf, ic_sup, alpha=0.2, color='#e74c3c', label='IC 95%')
ax.plot(['2025-S1','2025-S2'], y_fut, 'D', color='#e74c3c', ms=9)
ax.axvline(x=9, color='gray', ls=':', lw=1.5)
ax.set_title('Projeção Ativo Total — Banco Master (R$ bi)')
ax.tick_params(axis='x', rotation=45); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()


## 5. Conclusões

1. **Crescimento extraordinário:** Ativos expandiram **668%** em 4 anos (R$ 8,2 bi → R$ 63 bi).
2. **Pressão no Basileia:** Índice recuou de 14,6% para **11,9%** em dez/2024, próximo do mínimo regulatório de 11%.
3. **Risco de liquidez:** CDB com taxas até 140% CDI e vencimento concentrado em jun/2025.
4. **Lucratividade:** Lucro dobrou para R$ 1,07 bi em 2024, mas sustentado por componentes menos recorrentes (cessões de carteira, TVM).
5. **Predição:** Regressão linear projeta ativo de R$ 68–75 bi em 2025 (R² = 0,97), mas aquisição pelo BRB altera estruturalmente a trajetória.
